# SMS TGNN-DDPG Major Revision
Drive-backed, restart-safe execution. Run cells in order. The notebook contains no experiment logic; it only calls the versioned CLI.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, subprocess
REPO_URL = 'https://github.com/sehyun00/sms-tgnn-ddpg-repro.git'
BRANCH = 'main'
WORKSPACE = pathlib.Path('/content/sms-tgnn-ddpg-repro')
if WORKSPACE.exists():
    subprocess.run(['git', '-C', str(WORKSPACE), 'fetch', 'origin', '--prune'], check=True)
    subprocess.run(['git', '-C', str(WORKSPACE), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(WORKSPACE), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(WORKSPACE)], check=True)
os.chdir(WORKSPACE)
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/sms_revision_runs/paper-v1')
os.environ['SMS_PREPARED_DIR'] = str(DRIVE_ROOT / 'prepared')
os.environ['SMS_RESULTS_DIR'] = str(DRIVE_ROOT / 'results')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

In [ ]:
import torch, platform
assert torch.cuda.is_available(), 'Enable a Colab GPU runtime before training.'
print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)})
subprocess.run(['python', '-m', 'pytest', '-q'], check=True)

In [ ]:
CLI = ['python', 'scripts/run_revision_experiments.py', '--config', 'config/paper_revision.yaml']
subprocess.run(CLI + ['--stage', 'prepare'], check=True)

In [ ]:
subprocess.run(CLI + ['--stage', 'train', '--fold', 'primary', '--model', 'all', '--seed', 'all', '--frequency', 'all', '--graph', 'primary', '--resume'], check=True)
subprocess.run(CLI + ['--stage', 'backtest', '--fold', 'primary', '--model', 'all', '--seed', 'all', '--frequency', 'all', '--graph', 'primary', '--universe', 'n10', '--resume'], check=True)

In [ ]:
subprocess.run(CLI + ['--stage', 'train', '--fold', 'primary', '--model', 'tgnn,hybrid', '--seed', 'all', '--frequency', 'all', '--graph', 'sector,correlation', '--resume'], check=True)
subprocess.run(CLI + ['--stage', 'backtest', '--fold', 'primary', '--model', 'tgnn,hybrid,hybrid_fixed', '--seed', 'all', '--frequency', 'quarterly', '--graph', 'all', '--universe', 'n10', '--resume'], check=True)

In [ ]:
subprocess.run(CLI + ['--stage', 'train', '--fold', 'secondary', '--model', 'all', '--seed', 'all', '--frequency', 'all', '--graph', 'primary', '--resume'], check=True)
subprocess.run(CLI + ['--stage', 'backtest', '--fold', 'secondary', '--model', 'all', '--seed', 'all', '--frequency', 'all', '--graph', 'primary', '--universe', 'n10', '--resume'], check=True)

In [ ]:
for universe in ['n5', 'n15']:
    subprocess.run(CLI + ['--stage', 'backtest', '--fold', 'primary', '--model', 'all', '--seed', '42', '--frequency', 'quarterly', '--graph', 'primary', '--universe', universe, '--resume'], check=True)

In [ ]:
subprocess.run(CLI + ['--stage', 'aggregate'], check=True)
aggregate = pathlib.Path(os.environ['SMS_RESULTS_DIR']) / 'aggregate'
print('Evidence:', aggregate)
print((aggregate / 'reviewer_response_evidence.md').read_text())